In [1]:
import csv
import time
from pathlib import Path
from urllib.parse import urljoin

import pymysql
import requests
from bs4 import BeautifulSoup

BASE_URL = "http://www.bspider.top/anjuke/"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}
FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

r = requests.get(BASE_URL, headers=HEADERS, timeout=30)
r.raise_for_status()
r.encoding = "utf-8"   # 页面本身就是 utf-8；apparent_encoding 会误判成 GBK 导致乱码
soup = BeautifulSoup(r.text, "html.parser")

print("状态码:", r.status_code)
print("页面标题:", soup.title.get_text(strip=True))
rows = soup.select("div.property")
print("本页房源数:", len(rows))


状态码: 200
页面标题: 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客
本页房源数: 60


In [2]:
row = rows[0]   # 只看第一条，逐个字段核对

print("title  :", row.select_one("h3.property-content-title-name").get_text(strip=True))
print("house  :", row.select_one("p.property-content-info-comm-name").get_text(strip=True))
print("address:", row.select_one("p.property-content-info-comm-address").get_text("", strip=True))
print("struct :", row.select_one("p.property-content-info-attribute").get_text("", strip=True))
print("price  :", row.select_one("p.property-price-average").get_text(strip=True))
print("属性项（顺序会变，需按特征判断）:")
for p in row.select("div.property-content-info")[0].select("p.property-content-info-text"):
    print("   ", repr(p.get_text("", strip=True)))


title  : 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house  : 万科金域学府翰林菁华里
address: 南岸学府大道二塘路59号
struct : 3室2厅1卫
price  : 16465元/㎡
属性项（顺序会变，需按特征判断）:
    '3室2厅1卫'
    '89.9㎡'
    '南北'
    '低层(共27层)'
    '2019年建造'


In [3]:
def parse_row(row):
    """从一条 div.property 中取出 8 个字段"""
    def text_of(selector):
        node = row.select_one(selector)
        return node.get_text("", strip=True) if node else ""

    area = floor = maketime = ""
    info = row.select("div.property-content-info")
    if info:
        for p in info[0].select("p.property-content-info-text"):
            t = p.get_text("", strip=True)
            if not t:
                continue
            if "㎡" in t or "平米" in t:
                area = t
            elif "年建" in t:
                maketime = t
            elif "层" in t:
                floor = t

    return {
        "title": text_of("h3.property-content-title-name"),
        "house": text_of("p.property-content-info-comm-name"),
        "address": text_of("p.property-content-info-comm-address"),
        "struct": text_of("p.property-content-info-attribute"),
        "area": area,
        "floor": floor,
        "maketime": maketime,
        "price": text_of("p.property-price-average"),
    }


for label, rec in (("第一条", parse_row(rows[0])), ("最后一条", parse_row(rows[-1]))):
    print(label)
    for f in FIELDS:
        print(f"   {f:<9}{rec[f]}")
    print()


第一条
   title    万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
   house    万科金域学府翰林菁华里
   address  南岸学府大道二塘路59号
   struct   3室2厅1卫
   area     89.9㎡
   floor    低层(共27层)
   maketime 2019年建造
   price    16465元/㎡

最后一条
   title    精装修 电梯房 4室2厅 91平
   house    华润凯旋天地
   address  南岸弹子石腾龙大道45号
   struct   4室2厅2卫
   area     91.3㎡
   floor    低层(共32层)
   maketime 2015年建造
   price    15886元/㎡



In [4]:
MAX_PAGES = 20

all_records = []
seen = set()
url = BASE_URL
stop_reason = ""

for page in range(1, MAX_PAGES + 1):
    if url in seen:
        stop_reason = f"地址重复，异常中止：{url}"
        break
    seen.add(url)

    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    resp.encoding = "utf-8"
    page_soup = BeautifulSoup(resp.text, "html.parser")
    page_rows = page_soup.select("div.property")
    if not page_rows:
        stop_reason = f"第 {page} 页没有房源，异常中止"
        break

    for item in page_rows:
        all_records.append(parse_row(item))

    nxt = page_soup.select_one("a.next.next-active")
    href = nxt.get("href") if nxt else None
    print(f"第 {page} 页  本页 {len(page_rows):>2} 条  累计 {len(all_records):>3} 条  url={url}")

    if not href:
        stop_reason = "下一页按钮不存在，已到末页"
        break
    url = urljoin(url, href)
    time.sleep(1.2)
else:
    stop_reason = f"达到安全页数上限 {MAX_PAGES}，中止（不算采集完成）"

print("停止原因:", stop_reason)
print("采集总条数:", len(all_records))


第 1 页  本页 60 条  累计  60 条  url=http://www.bspider.top/anjuke/
第 2 页  本页 60 条  累计 120 条  url=http://www.bspider.top/anjuke/?page=2
第 3 页  本页 60 条  累计 180 条  url=http://www.bspider.top/anjuke/?page=3
第 4 页  本页 60 条  累计 240 条  url=http://www.bspider.top/anjuke/?page=4
第 5 页  本页 60 条  累计 300 条  url=http://www.bspider.top/anjuke/?page=5
第 6 页  本页 60 条  累计 360 条  url=http://www.bspider.top/anjuke/?page=6
第 7 页  本页 60 条  累计 420 条  url=http://www.bspider.top/anjuke/?page=7
第 8 页  本页 60 条  累计 480 条  url=http://www.bspider.top/anjuke/?page=8
第 9 页  本页 60 条  累计 540 条  url=http://www.bspider.top/anjuke/?page=9
第 10 页  本页 60 条  累计 600 条  url=http://www.bspider.top/anjuke/?page=10
停止原因: 下一页按钮不存在，已到末页
采集总条数: 600


In [5]:
unique = []
dedup_seen = set()
for rec in all_records:
    key = tuple(rec[f] for f in FIELDS)
    if key not in dedup_seen:
        dedup_seen.add(key)
        unique.append(rec)

print("采集去重前:", len(all_records))
print("完全相同重复:", len(all_records) - len(unique))
print("去重后:", len(unique))

Path("data").mkdir(exist_ok=True)
csv_path = Path("data") / "houseinfo.csv"
with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(FIELDS)
    for rec in unique:
        writer.writerow([rec[f] for f in FIELDS])
print("已保存:", csv_path.resolve())


采集去重前: 600
完全相同重复: 102
去重后: 498
已保存: /home/pwuser/workspace/lab02/data/houseinfo.csv


In [6]:
with open(csv_path, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("表头:", reader.fieldnames)
    csv_rows = list(reader)

print("CSV 数据行数（不含表头）:", len(csv_rows))
for tag, i in (("第一条", 0), ("跨页一条", 60), ("最后一条", len(csv_rows) - 1)):
    rec = csv_rows[i]
    print(f"\n{tag}（第 {i + 1} 行）")
    for f in FIELDS:
        print(f"   {f:<9}{rec[f]}")


表头: ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
CSV 数据行数（不含表头）: 498

第一条（第 1 行）
   title    万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
   house    万科金域学府翰林菁华里
   address  南岸学府大道二塘路59号
   struct   3室2厅1卫
   area     89.9㎡
   floor    低层(共27层)
   maketime 2019年建造
   price    16465元/㎡

跨页一条（第 61 行）
   title    万科物业自营 朝中庭 业主诚心出售
   house    万科金域学府翰林菁华里
   address  南岸学府大道二塘路59号
   struct   3室2厅1卫
   area     85.7㎡
   floor    低层(共27层)
   maketime 2018年建造
   price    16914元/㎡

最后一条（第 498 行）
   title    恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售
   house    恒大照母山山水城
   address  渝北照母山恒康路9号
   struct   3室2厅1卫
   area     90.2㎡
   floor    低层(共32层)
   maketime 2015年建造
   price    15530元/㎡


In [10]:
conn = pymysql.connect(
    host="mysql", port=3306, user="student", password="123456",
    database="smartpark_lab", charset="utf8mb4", autocommit=False,
)
cur = conn.cursor()
cur.execute("SELECT VERSION(), DATABASE(), CURRENT_USER()")
print(cur.fetchone())

cur.execute("""
CREATE TABLE IF NOT EXISTS houseinfo (
    id INT AUTO_INCREMENT PRIMARY KEY COMMENT '自增长主键',
    title VARCHAR(200) COMMENT '标题',
    house VARCHAR(50) COMMENT '楼盘',
    address VARCHAR(200) COMMENT '地址',
    struct VARCHAR(50) COMMENT '户型',
    area VARCHAR(20) COMMENT '建筑面积',
    floor VARCHAR(20) COMMENT '楼层',
    maketime VARCHAR(20) COMMENT '建造年代',
    price VARCHAR(20) COMMENT '每平方米单价'
) CHARACTER SET utf8mb4
""")
conn.commit()
cur.execute("SELECT COUNT(*) FROM houseinfo")
print("入库前条数:", cur.fetchone()[0])
cur.close()
conn.close()


('8.4.11', 'smartpark_lab', 'student@%')
入库前条数: 498


In [11]:
with open(csv_path, encoding="utf-8-sig", newline="") as f:
    csv_records = [tuple(row[f] for f in FIELDS) for row in csv.DictReader(f)]

INSERT_SQL = (
    "INSERT INTO houseinfo (title, house, address, struct, area, floor, maketime, price) "
    "VALUES (%s, %s, %s, %s, %s, %s, %s, %s)"
)

conn = None
cur = None
try:
    conn = pymysql.connect(
        host="mysql", port=3306, user="student", password="123456",
        database="smartpark_lab", charset="utf8mb4", autocommit=False,
    )
    cur = conn.cursor()
    cur.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo")
    existing = {tuple("" if v is None else str(v) for v in row) for row in cur.fetchall()}
    pending = [row for row in csv_records if row not in existing]
    print(f"CSV {len(csv_records)} 条 / 库内已有 {len(existing)} 条 / 本次待插入 {len(pending)} 条")

    if pending:
        affected = cur.executemany(INSERT_SQL, pending)
        conn.commit()
        print("提交成功，插入", affected, "条")
    else:
        print("没有新记录，跳过插入（防止重复运行插两遍）")
except Exception as exc:
    print("入库失败，执行回滚:", exc)
    if conn is not None:
        conn.rollback()
    raise
finally:
    if cur is not None:
        cur.close()
    if conn is not None:
        conn.close()
    print("游标与连接已关闭")


CSV 498 条 / 库内已有 498 条 / 本次待插入 0 条
没有新记录，跳过插入（防止重复运行插两遍）
游标与连接已关闭


In [9]:
conn = pymysql.connect(
    host="mysql", port=3306, user="student", password="123456",
    database="smartpark_lab", charset="utf8mb4", autocommit=True,
)
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM houseinfo")
print("入库后条数:", cur.fetchone()[0])

cur.execute("SELECT id, title, house, address, struct, area, floor, maketime, price "
            "FROM houseinfo ORDER BY id LIMIT 5")
for row in cur.fetchall():
    print(row)
cur.close()
conn.close()


入库后条数: 498
(2, '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡')
(3, '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '两代一家', '渝北渝北黄泥磅紫荆路29号', '3室2厅2卫', '94.5㎡', '低层(共28层)', '2008年建造', '14815元/㎡')
(4, '鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售', '大雅云居山', '渝北鸳鸯龙展路59号', '3室2厅1卫', '88㎡', '低层(共24层)', '2016年建造', '16487元/㎡')
(5, '中交中央公园 横厅精装大两房 出门都是公园', '中交中央公园悦颂', '渝北中央公园翠丹路4号', '2室2厅1卫', '91㎡', '低层(共25层)', '2016年建造', '20660元/㎡')
(6, '富力现代广场 豪装大两房 户型方正 看房方便', '富力现代广场', '南岸弹子石弹子石新街59号', '2室2厅1卫', '85.5㎡', '低层(共30层)', '2012年建造', '16969元/㎡')
